Panda version of the sample code

Examine the json reponse of keys

In [0]:
%skip
import requests as requests

url = "https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&starttime=2014-01-01&endtime=2014-01-02"
response = requests.get(url)
data = response.json()
print(type(data))           # <class 'dict'>
print(data.keys())          # dict_keys(['type', 'metadata', 'features'])
print(len(data["features"])) # 100+ events

Level1 

In [0]:
%skip
import pandas as pd
import requests as request
from datetime import datetime, timedelta
 

url = "https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&starttime=2014-01-01&endtime=2014-01-02"


def get_data(url):
    response = requests.get(url)  # Fix: requests, not request
    data = response.json()
    return pd.json_normalize(data["features"])  
df_earthquake_catalog = get_data(url)
display(df_earthquake_catalog)
#df_earthquake_catalog.head()
 

In [0]:


import pandas as pd
import requests as requests
from datetime import datetime, timedelta
import re

def get_earthquake_data(start_date, end_date):
    """
    Fetch earthquake data for date range
    Args:
        start_date: str or datetime (e.g., "2014-01-01" or datetime(2014, 1, 1))
        end_date: str or datetime
    """
    # Convert datetime to string if needed
    if isinstance(start_date, datetime):
        start_date = start_date.strftime("%Y-%m-%d")
    if isinstance(end_date, datetime):
        end_date = end_date.strftime("%Y-%m-%d")
    
    # Build dynamic URL
    url = f"https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&starttime={start_date}&endtime={end_date}"
    
    # Fetch and parse
    response = requests.get(url)
    data = response.json()
    #df = pd.json_normalize([f["properties"] for f in data["features"]])
    #df = pd.json_normalize([f["geometry"] for f in data["features"]])
    df = pd.json_normalize(data["features"])

    # Extract features (actual earthquake records)
    features = data["features"]

    # Add geometry columns (lat/lon)
    coords = [feature["geometry"]["coordinates"] for feature in features]
    df["geometry_longitude"] = [coord[0] for coord in coords]
    df["geometry_latitude"] = [coord[1] for coord in coords]
    df["geometry_depth"] = [coord[2] for coord in coords]

    
    # Clean column names (remove 'properties.' prefix if exists)
    df.columns = [col.replace('properties.', 'properties_') for col in df.columns]
    df.columns = [col.replace('geometry.', 'geometry_') for col in df.columns]
    #df.columns = [re.sub(r'properties\.|geometry\.', '', col) for col in df.columns]
    return df

# Current datetime
now = datetime.now()

# Previous day (subtract 1 day)
next_day = now + timedelta(days=1)
print(now, next_day)
df_earthquake_catalog_l2 = get_earthquake_data(now, next_day)
display(df_earthquake_catalog_l2)


insert into catalog

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, IntegerType , ArrayType

spark = SparkSession.builder.getOrCreate()

# Define schema
schema = StructType([
    StructField("type", StringType(), True),
    StructField("id", StringType(), True),
    StructField("properties_mag", DoubleType(), True),
    StructField("properties_place", StringType(), True),
    StructField("properties_time", LongType(), True),
    StructField("properties_updated", LongType(), True),
    StructField("properties_tz", StringType(), True),
    StructField("properties_url", StringType(), True),
    StructField("properties_detail", StringType(), True),
    StructField("properties_felt", DoubleType(), True),
    StructField("properties_cdi", DoubleType(), True),
    StructField("properties_mmi", DoubleType(), True),
    StructField("properties_alert", StringType(), True),
    StructField("properties_status", StringType(), True),
    StructField("properties_tsunami", IntegerType(), True),
    StructField("properties_sig", IntegerType(), True),
    StructField("properties_net", StringType(), True),
    StructField("properties_code", StringType(), True),
    StructField("properties_ids", StringType(), True),
    StructField("properties_sources", StringType(), True),
    StructField("properties_types", StringType(), True),
    StructField("properties_nst", LongType(), True),
    StructField("properties_dmin", DoubleType(), True),
    StructField("properties_rms", DoubleType(), True),
    StructField("properties_gap", LongType(), True),
    StructField("properties_magType", StringType(), True),
    StructField("properties_type", StringType(), True),
    StructField("properties_title", StringType(), True),
    StructField("geometry_type", StringType(), True),
    StructField("geometry_coordinates", ArrayType(DoubleType()), True),
    StructField("geometry_longitude", DoubleType(), True),
    StructField("geometry_latitude", DoubleType(), True),
    StructField("geometry_depth", DoubleType(), True)
])

# Convert pandas to Spark
spark_df = spark.createDataFrame(df_earthquake_catalog_l2, schema=schema)

table_name = "labuser_1_dev.default.earthquake_catalog_bronze"

# Perform upsert
if spark.catalog.tableExists(table_name):
    delta_table = DeltaTable.forName(spark, table_name)
    
    (delta_table.alias("target")
        .merge(spark_df.alias("source"), "target.id = source.id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f"Merged data into {table_name}")
else:
    spark_df.write.format("delta").saveAsTable(table_name)
    print(f"Created new table {table_name}")